# 02 — The instrument: a panel whose truth is known

No real advertiser knows the true ROI of its own media plan. That is the problem
the whole field is organised around, and it means "did the model recover the
truth?" cannot be asked of real data at all.

So it is asked here instead, of a panel constructed for the purpose, from a
configuration committed before any model was fitted. **This is an instrument, not
a substitute for data that was unavailable.** Nothing it produces describes the
effectiveness of a real marketing channel.

In [1]:
import warnings

import numpy as np
import pandas as pd

from athar import paths
from athar.provenance import read_metric

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

METRICS = paths.metrics_dir()
PROCESSED = paths.processed_dir()


def show(frame, caption=""):
    if caption:
        print(caption)
    print(frame.to_string(index=False))
    print()

In [2]:
from athar import dgp

config = dgp.load_config()
panel_metrics = read_metric("panel", METRICS)
print("configuration digest:", config.digest)
print("seed:", config.spec["seed"])
print()
print(panel_metrics["provenance"]["caveat"])

configuration digest: d9e7b488eb749462
seed: 20260829

SYNTHETIC — the channel-spend panel and every media effect in it are simulated from a pre-registered data-generating process, not measured. The revenue baseline is real Olist history; the media contribution layered on it is not. No figure here describes the effectiveness of any real marketing channel, and none may be quoted as one. What is measured is whether a method recovers a truth that is known only because it was constructed.


## The generating process

Spend is generated in logs from two latent AR(1) factors — budget pressure, which
moves every channel together, and funnel tilt, which trades performance spend
against brand and upper-funnel spend. Two factors rather than one because a single
factor can only produce a rank-one correlation structure, in which every pair of
channels correlates in the same direction, which is not what a media plan looks
like.

Effect is delayed-geometric adstock followed by a Hill curve. The coefficient on
each channel is solved so its average ROI lands exactly on the pre-registered
target.

In [3]:
show(pd.DataFrame(panel_metrics["channels"])[[
    "channel", "mean_weekly_spend", "true_roi_average", "true_roi_marginal",
    "adstock_alpha", "adstock_theta", "adstock_peak_lag_weeks", "hill_slope",
]], "The five channels as configured")

The five channels as configured
        channel  mean_weekly_spend  true_roi_average  true_roi_marginal  adstock_alpha  adstock_theta  adstock_peak_lag_weeks  hill_slope
   search_brand             4000.0               1.6           0.957632            0.3            0.0                       0         1.6
search_nonbrand             7000.0               2.8           1.901297            0.4            0.0                       0         1.2
    social_paid             6000.0               2.1           1.483514            0.6            1.0                       1         1.4
   display_prog             3500.0               0.9           0.580361            0.7            2.0                       2         1.0
      video_ctv             4500.0               2.5           2.783987            0.8            3.0                       3         1.8



## Average ROI is not marginal ROI, and the difference is not a constant

Average ROI divides total contribution by total spend, and is what a media-mix
model usually reports. Marginal ROI is the slope at the current plan, and is what
a reallocation decision turns on.

Which is larger is *not* fixed. A Hill curve with a slope above one is S-shaped:
below its inflection the response is convex, marginal exceeds average, and the
channel is under-invested. Above it, the curve is concave and marginal falls
below average. Both regimes are present here at the configured spend, which is
what makes notebook 08 an allocation problem rather than a ranking exercise.

In [4]:
channels = pd.DataFrame(panel_metrics["channels"])
channels["regime"] = np.where(
    channels["true_roi_marginal"] > channels["true_roi_average"],
    "convex — under-invested", "concave — saturating")
show(channels[["channel", "true_roi_average", "true_roi_marginal",
               "marginal_over_average", "regime"]])

        channel  true_roi_average  true_roi_marginal  marginal_over_average                  regime
   search_brand               1.6           0.957632               0.598520    concave — saturating
search_nonbrand               2.8           1.901297               0.679035    concave — saturating
    social_paid               2.1           1.483514               0.706435    concave — saturating
   display_prog               0.9           0.580361               0.644845    concave — saturating
      video_ctv               2.5           2.783987               1.113595 convex — under-invested



## The deliberate misspecification

The panel is generated with a kernel that peaks at a positive lag. The headline
model in notebook 03 is fitted with plain geometric adstock, which is the same
kernel with its delay pinned to zero and therefore cannot express a delayed peak
at all, and with a logistic curve that cannot take the Hill shape.

Fitting the generating form to its own output recovers the assumptions and
measures nothing. A matched arm runs as a control so the two sources of error can
be told apart.

One practical note that shaped the design: the kernel was originally a Weibull
PDF. `WeibullPDFAdstock` raises `TypeError: x must be have an XTensorType` under
pymc-marketing 0.19.2 with pytensor 2.38.2, for every saturation. `WeibullCDFAdstock`
samples cleanly but decays monotonically and cannot represent a delayed peak
either. The delayed-geometric form both admits a delayed peak and can actually be
fitted, which is what makes the matched arm genuinely matched.

In [5]:
for key, value in panel_metrics["generating_specification"].items():
    print(f"{key}:\n  {value}\n")

adstock:
  Delayed geometric: w_l proportional to alpha**((l - theta)**2), normalised, which peaks at lag theta

fitted_with:
  Geometric adstock and logistic saturation. Geometric adstock is the generating kernel with theta pinned to zero, so it cannot represent a delayed peak at all, and a logistic curve cannot take the Hill shape. The mismatch is deliberate: a model fitted to data its own functional form produced recovers its own assumptions and measures nothing. The matched arm of the recovery grid separates misspecification error from identification error.

saturation:
  Hill



## How hard the identification problem is

Two numbers bound what any model can do here, and both are reported per cell of
the recovery grid in notebook 04.

The condition number and VIF say how separable the channels are. The media share
of *detrended* revenue variance says how much signal there is to separate — the
raw share is dominated by Olist's fivefold growth, which any model absorbs into
its own trend terms rather than having to explain with media.

In [6]:
ident = panel_metrics["identification"]
print("max pairwise correlation :", round(ident["max_pairwise_correlation"], 4))
print("condition number         :", round(ident["condition_number"], 3))
print("max VIF                  :", round(ident["max_vif"], 3))
print("media share of variance  :", round(ident["media_share_of_revenue_variance"], 4), "(raw)")
detrended = round(ident["media_share_of_detrended_variance"], 4)
print("                          ", detrended, "(detrended — the one to quote)")
print()
print(ident["note"])
print()
show(pd.DataFrame(ident["correlation"]).round(3).reset_index().rename(columns={"index": ""}),
     "Realised spend correlation")

max pairwise correlation : 0.8881
condition number         : 5.953
max VIF                  : 5.755
media share of variance  : 0.0706 (raw)
                           0.1723 (detrended — the one to quote)

The detrended share is the one that bounds identification. The raw share is dominated by Olist's fivefold growth across the window, which any media-mix model absorbs into its own trend and seasonality terms rather than having to explain with media.

Realised spend correlation
                 display_prog  search_brand  search_nonbrand  social_paid  video_ctv
   display_prog         1.000         0.818            0.582        0.612      0.563
   search_brand         0.818         1.000            0.622        0.690      0.617
search_nonbrand         0.582         0.622            1.000        0.888      0.254
    social_paid         0.612         0.690            0.888        1.000      0.389
      video_ctv         0.563         0.617            0.254        0.389      1.000



## Attribution, and its one pre-registered knob

Last-click here is a parametric caricature, not a simulated journey: a tracking
rate, the share of a channel's true contribution it observes at all, and an
organic capture rate, the share of *baseline* revenue it credits to that channel.
The second is the mechanism the whole project is about.

Simulating individual journeys would look more faithful without being more honest
— the journey parameters would be exactly as invented, only harder to state. The
cost of this choice is that no claim is made about attribution *mechanics*, only
about the consequences of a stated bias.

`search_nonbrand` carries a tracking rate of 1 and an organic capture of 0, so
last-click recovers its true ROI exactly. That case is in the design deliberately.

In [7]:
show(channels[["channel", "true_roi_average", "lastclick_roas",
               "lastclick_bias_relative", "tracking_rate", "organic_capture"]],
     "What last-click would report")
print("most overstated :", panel_metrics["attribution_summary"]["most_overstated"])
print("most understated:", panel_metrics["attribution_summary"]["most_understated"])
print()
print(panel_metrics["attribution_summary"]["null_case"])

What last-click would report
        channel  true_roi_average  lastclick_roas  lastclick_bias_relative  tracking_rate  organic_capture
   search_brand               1.6        7.473196                 3.670747           1.00            0.150
search_nonbrand               2.8        2.800000                 0.000000           1.00            0.000
    social_paid               2.1        4.083247                 0.944404           0.95            0.080
   display_prog               0.9        1.972445                 1.191605           0.70            0.030
      video_ctv               2.5        0.799021                -0.680392           0.25            0.005

most overstated : search_brand
most understated: video_ctv

search_nonbrand carries tracking_rate 1.0 and organic_capture 0.0, so last-click recovers its true ROI exactly. A harness that only ever showed attribution failing would have had its answer chosen for it.


## Verification: the stored truth is the panel's actual truth

Recomputed straight from the generated series, sharing no code path with the solve
that produced the coefficients. A drifted truth would make every recovery number in
notebooks 03 and 04 wrong in a way nothing downstream could detect.

In [8]:
for key, value in panel_metrics["verification"].items():
    print(f"{key:48s} {value}")

fitting_frame_columns                            ['display_prog', 'revenue', 'search_brand', 'search_nonbrand', 'social_paid', 'video_ctv', 'week']
fitting_frame_leaks_truth                        False
truth_reconciliation_worst_relative_difference   0.0
